# Demo - LoRA Parameter Math
**Day 3 - Session 2**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day3/demos/notebooks/demo-lora-parameters.ipynb)

**Goal:** Show why LoRA trains so few parameters and how a low-rank B x A
update approximates a full-size weight change.

Part 1 counts parameters: a full weight matrix W is d x k, while LoRA learns
two small matrices B (d x r) and A (r x k). For a small rank r the trainable
count drops by a large factor. Part 2 builds a real weight update, factors it
with an SVD-based rank-r approximation, and shows the reconstruction error
shrinks as r grows. That is the same idea LoRA relies on.

No API key, no downloads. Pure NumPy on CPU.

In [1]:
# Scoped install for this demo (safe to re-run; numpy is preinstalled on Colab).
%pip install -q numpy

In [2]:
import numpy as np

np.random.seed(0)

## 1. Parameter count: full matrix vs LoRA adapter
A full 4096 x 4096 projection has millions of parameters. LoRA's two small
matrices (B x A) train only a tiny fraction of that count.

In [3]:
def count_parameters():
    """Compare full-matrix parameters against LoRA B x A parameters."""
    d, k = 4096, 4096  # a typical attention projection shape
    full = d * k
    print(f"  Weight matrix W: {d} x {k} = {full:,} parameters")
    print(f"  {'rank r':>8}  {'LoRA params (B x A)':>20}  {'% of full':>10}")
    for r in (4, 8, 16, 64):
        lora = d * r + r * k  # B is d x r, A is r x k
        print(f"  {r:>8}  {lora:>20,}  {100 * lora / full:>9.2f}%")
    print("\n  A small rank trains a tiny fraction of the full weight count.")


count_parameters()

  Weight matrix W: 4096 x 4096 = 16,777,216 parameters
    rank r   LoRA params (B x A)   % of full
         4                32,768       0.20%
         8                65,536       0.39%
        16               131,072       0.78%
        64               524,288       3.12%

  A small rank trains a tiny fraction of the full weight count.


## 2. How well a rank-r update approximates a full change
Take an "ideal" full weight update, factor it with an SVD, and keep only the
top r components. The reconstruction error shrinks as r grows.

In [4]:
def low_rank_approximation():
    """Show a rank-r B x A approximates a full weight update."""
    d, k = 64, 64
    delta_w = np.random.randn(d, k)  # the "ideal" full-fine-tune weight update
    u, s, vt = np.linalg.svd(delta_w)
    total_energy = np.sum(s)

    print(f"  Target update DeltaW: {d} x {k}")
    print(f"  {'rank r':>8}  {'reconstruction error':>22}  {'energy kept':>12}")
    for r in (1, 4, 8, 32, 64):
        # Best rank-r factorization: B = U[:, :r] * S[:r], A = Vt[:r]
        b = u[:, :r] * s[:r]
        a = vt[:r]
        approx = b @ a
        error = np.linalg.norm(delta_w - approx) / np.linalg.norm(delta_w)
        kept = np.sum(s[:r]) / total_energy
        print(f"  {r:>8}  {error:>22.3f}  {kept:>11.1%}")

    print("\n  Higher rank captures more of the update at the cost of more params.")


low_rank_approximation()

  Target update DeltaW: 64 x 64
    rank r    reconstruction error   energy kept
         1                   0.971         3.5%
         4                   0.890        13.4%
         8                   0.793        25.2%
        32                   0.332        76.1%
        64                   0.000       100.0%

  Higher rank captures more of the update at the cost of more params.


## Expected output
- Part 1: even rank 64 trains only a few percent of the full 16.7M parameters.
- Part 2: reconstruction error falls as rank rises, reaching 0 at full rank.

**Takeaway:** low rank captures most of a weight change for few parameters.